# Illustrer la tomographie

In [ ]:
import random
from copy import deepcopy

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.ndimage import convolve

from tomography import random_discrete_image
from tomography import random_discrete_image_3d
from tomography.tomographic_reconstruction import radon_transform

pio.renderers.default = "browser"  # plotly_mimetype // browser

COLORSCALE: str = "Viridis"

## Illustration 3D

### Illustration d'un organe fictif


In [ ]:
S = 40  # 40
np.random.seed(S)
random.seed(S)

_N = 75
V = random_discrete_image_3d(_N, _N, _N, 20)
CS = 5
kernel = np.ones((CS, CS, CS), dtype=float) / (CS ** 3)
V = convolve(V, kernel, mode='reflect')

X, Y, Z = np.meshgrid(np.linspace(-1, 1, _N), np.linspace(-1, 1, _N), np.linspace(-1, 1, _N))

fig = go.Figure(data=go.Volume(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=V.flatten(),
    isomin=0.1,
    isomax=1.0,
    opacity=0.15,
    surface_count=5,
    colorscale=COLORSCALE,
    showscale=False,
))


fig.update_yaxes(showticklabels=False, visible=False)

fig.update_layout(
    template='plotly_white',
    scene_camera=dict(
        up=dict(x=0, y=0, z=-1),
        center=dict(x=-0, y=0, z=-0.1),
        eye=dict(x=1.2, y=1.2, z=1.2)
    ),
    margin=dict(
        l=0,
        r=0,
        b=0,
        t=0,
    ),
    autosize=True,
    scene=dict(
        xaxis=dict(
            showticklabels=False,
            title={"text": ""},
            range=[-1, 1],
            tick0=0,
            dtick=0.2,
        ),
        yaxis=dict(
            showticklabels=False,
            title={"text": ""},
            range=[-1, 1],
            tick0=0,
            dtick=0.2,
        ),
        zaxis=dict(
            showticklabels=False,
            title={"text": ""},
            range=[-1, 1],
            tick0=0,
            dtick=0.2,
        ),
    )
)

# fig.show()

fig.write_image("images/1 fig0 lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/1 fig0 hr.png", scale=13, height=600, width=600)

### Illustration de ce que va faire la caméra

In [ ]:
cx, cy, cz = 0, 0, 0   # cercle sur le plan z=0
r = 1 # rayon du cercle

theta = np.linspace(0, 2 * np.pi, 200)
x_circle = cx + r * np.cos(theta)
y_circle = cy + r * np.sin(theta)
z_circle = np.full_like(x_circle, cz)

fig.add_trace(go.Scatter3d(
    x=x_circle,
    y=y_circle,
    z=z_circle,
    mode='lines',
    line=dict(color='black', width=7.5),
    name='Cercle',
    showlegend=False
))

# fig.show()

fig.write_image("images/1 fig1 lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/1 fig1 hr.png", scale=13 / 4, height=600 * 4, width=600 * 4)


## Idée générale

### Illustration fictive de densité osseuse

In [ ]:

S = 65  # 55 65
np.random.seed(S)
random.seed(S)

image = random_discrete_image(1000, 1000, 20)

empty_img = deepcopy(image)
empty_img[empty_img == 0] = np.nan

fig = go.Figure(data=go.Heatmap(
    z=empty_img,
    colorscale=COLORSCALE,
    colorbar=dict(
        title="Densité",
    ),
)
)

fig.update_xaxes(scaleanchor='x',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks="",
                 showline=True,
                 showgrid=False,
                 scaleratio=1,
                 showticklabels=False)
fig.update_yaxes(scaleanchor='y',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks="",
                 showline=True,
                 showgrid=False,
                 scaleratio=1,
                 showticklabels=False,
                 )

fig.update_layout(
    template='plotly_white',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.025,
        bordercolor="black",
        borderwidth=1),
    title={"text": r"$\LARGE{\text{Simulation d'un densité osseuse}}$"},
)

# fig.show()

fig.write_image("images/1 fig2 lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/1 fig2 hr.png", scale=13, height=600, width=600)

### Illustration du sinogramme

In [ ]:
rad = radon_transform(image, np.linspace(0, 360, 1000))

fig = go.Figure(
    data=go.Heatmap(
        x=np.linspace(0, 360, 1000),
        z=rad,
        colorscale=COLORSCALE,
        showscale=False,
    )
)

fig.update_xaxes(scaleanchor='x',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks='outside',
                 showline=True,
                 showgrid=False,
                 scaleratio=1)
fig.update_yaxes(scaleanchor='y',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks=None,
                 showline=True,
                 showgrid=False,
                 scaleratio=1,
                 showticklabels=False,)

fig.update_layout(
    template='plotly_white',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.025,
        bordercolor="black",
        borderwidth=1),
    title={"text": r"$\LARGE{\text{Sinogramme de la densité osseuse}}$"},
    xaxis={
        "title": {"text": r"$\Large{\text{Angles de vu en degré }}$"},
        "tickvals": np.linspace(0, 360, 10),
        "ticktext": [f"{int(v)}°" for v in np.linspace(0, 360, 10)]
    }
)

# fig.show()

fig.write_image("images/1 fig3 lr.webp", scale=1.3 * 2, height=600, width=600)
fig.write_image("images/1 fig3 hr.png", scale=13, height=600, width=600)